In [ ]:
!nvidia-smi

Tue Jun  2 22:01:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/kriteenjain/COMSCI260c-project.git"
REPO_DIR = "/content/COMSCI260c-project"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls

cwd: /content/COMSCI260c-project
notebooks  requirements.txt  run_baseline.py	       run_qwen_compression.py
README.md  results	     run_llama_compression.py  src


In [ ]:
!pip install -q -r requirements.txt
!pip install -q "autoawq>=0.2.6"
!pip install -q tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from huggingface_hub import login, whoami
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    login(token=token)
except Exception:
    login()

os.environ["HF_TOKEN"] = token if 'token' in dir() else ""
os.environ["HF_HOME"] = "/content/.hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(whoami()["name"], "— authenticated ✓")


seanhoffy — authenticated ✓


In [ ]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
TASK = "both"
LIMIT = 100
BATCH_SIZE = 1                                    # 1 for LLaMA 3B on T4
COMPRESSED_DIR = "/content/compressed_llama"     # separate from Qwen cache

import os
os.makedirs(COMPRESSED_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)
print({"MODEL": MODEL, "TASK": TASK, "LIMIT": LIMIT, "BATCH_SIZE": BATCH_SIZE})

{'MODEL': 'meta-llama/Llama-3.2-3B-Instruct', 'TASK': 'both', 'LIMIT': 100, 'BATCH_SIZE': 1}


In [ ]:
WANDA_U50_DIR = f"{COMPRESSED_DIR}/llama-wanda-u50"

!python run_llama_compression.py \
    --model {MODEL} \
    --method wanda \
    --sparsity-type unstructured \
    --sparsity-ratio 0.5 \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --save-compressed {WANDA_U50_DIR}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
[load] meta-llama/Llama-3.2-3B-Instruct
config.json: 100% 878/878 [00:00<00:00, 3.84MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 24.6MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:01<00:00, 7.16MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.70MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 20.9k/20.9k [00:00<00:00, 12.3MB/s]
Fetching 2 files: 100% 2/2 [00:16<00:00,  8.47s/it]
Download complete: 100% 6.43G/6.43G [00:17<00:00, 378MB/s]
Loading weights: 100% 254/254 [00:00<00:00, 903.16it/s, Material

In [ ]:
AWQ_DIR = f"{COMPRESSED_DIR}/llama-awq-w4g128"

!python run_llama_compression.py \
    --model {MODEL} \
    --method awq \
    --w-bit 4 \
    --q-group-size 128 \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --save-compressed {AWQ_DIR}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, 